# Day 6 poses the question.... Is BAAI/bge-m3 Actually The right embedding model? A Real Hit-Rate Benchmark

Before touching anything, one honest thing worth saying plainly: today's job is to test whether my embedding model needs improving,
not to assume it does. *My Month 2 Day 7* notebook already used **BAAI/bge-m3**, a genuinely strong, modern embedding model,
not a weak placeholder. So today isn't "*fix the embeddings*" it's "find out, with real numbers, whether fixing them would even help," which is exactly the discipline this whole month has been built around.There is no real gap in my own prior work worth fixing along the way. Looking back at Day 3's `generate_qa_pair` loop: chunk["content"], the exact source text a question was generated from
was sitting right there in memory, but only chunk["section"] ever got attatched to the saved pair. That's fine for Answer Correctness *(which just needs a ground-truth string)*, but it's useless for what today needs: Knowing exactly which chunk should be retrieved, so you can check whether it actually was.

# Coding!: A Chunk -> Traceable Benchmark Set + Hit Rate@K

**Fixing the Traceability Gap:** Re-running a small version of Day 3's generator, this time explicitly saving the exact source chunk text alongside each question, the one thing Day 3 dropped.

**Building a controlled Comparison Pool:** Rather than rebuilding my whole MongoDB Atlas index for every candidate embedding model *(expensive and slow)*, embed a fixed pool of real chunks locally with each candidate model and rank them by cosine similarity, a lightweight, honest way to compare models head to head on identical data.

**Compute hit rate@4:** For every benchmark question, check whether it's true source chunk lands in the top 4 results, matching my actual production `similarity_top_k = 4`, for each candidate model


In [ ]:
# setting up vector db connection before i set up the vector engine 
from groq import Groq
from dataclasses import dataclass, field, asdict
from sentence_transformers import SentenceTransformer
import numpy as np
from dotenv import load_dotenv, find_dotenv
import os
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(
    model_name= 'BAAI/bge-m3'
)

load_dotenv(find_dotenv())
client = Groq()

# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)     


# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)



print('🤖🛩️ Vector Store connection established ⚡')

🤖🛩️ Vector Store connection established ⚡


In [4]:
RELEVANCE_THRESHOLD = 0.5

def retrieve_with_confidence(query: str) -> tuple[list, float]:
    """Returns retrieved docs and the top chunk's confidence score.
    Uses cosine similarity to score from your vectore store.
    """

    # creating a native llamaindex retriever from my initialized index
    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)

    # reults is a list of (document, score) tuples
    # lower score = more similar in FAISS  (L2 distance); invert if needed
    # For Cosine similarity stores, higher = better

    if not results:
        return [], 0.0
    
    
    top_score = float(results[0].score) if results[0].score is not None else 0.0

    print(f"📣 Top retrieval score : {top_score:.3f} (threshold: {RELEVANCE_THRESHOLD})")
    return results, top_score

In [5]:
# PULLING CHUNKS SPREAD ACROSS THE DOCUMENT
# TOKEN USER TRACKER

@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()


SECTION_QUERIES = [
    "business overview and products",
    "risk factors",
    "management discussion and analysis",
    "consolidated statements of operations",
    "consolidated balance sheet",
    "notes to financial statements",
]

def sample_chunks_across_sections(chunks_per_section: int = 3) -> list[dict]:
    sampled = []
    for section_query in SECTION_QUERIES:
        docs, score = retrieve_with_confidence(section_query)
        for doc in docs[:chunks_per_section]:
            sampled.append({
                "section": section_query,
                "content": doc.node.get_content() if hasattr(doc, "node") else doc.page_content,
                "retrieval_score": score,
            })
    print(f"📋 Sampled {len(sampled)} chunks across {len(SECTION_QUERIES)} sections")
    return sampled

sampled_chunks = sample_chunks_across_sections(chunks_per_section=3)

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

📣 Top retrieval score : 0.798 (threshold: 0.5)
📣 Top retrieval score : 0.813 (threshold: 0.5)
📣 Top retrieval score : 0.775 (threshold: 0.5)
📣 Top retrieval score : 0.828 (threshold: 0.5)
📣 Top retrieval score : 0.829 (threshold: 0.5)
📣 Top retrieval score : 0.797 (threshold: 0.5)
📋 Sampled 18 chunks across 6 sections


In [6]:
# Fixing Day 3's gap, Generating aSmall, Chunk Traceable Benchmark Set

import re
import json
from typing import Optional

CHUNK_TRACEABLE_GENERATOR_PROMPT = """You are generating an evaluation question for a
RAG system testing an Apple FY2024 10-K.

You will receive ONE chunk of real text. Write exactly one question answerable COMPLETELY
from this chunk alone, plus the correct answer.

Respond in EXACTLY this format:
QUESTION: <the question, or SKIP>
ANSWER: <the ground truth answer, or SKIP>
"""

def generate_traceable_pair(chunk: dict) -> Optional[dict]:
    raw = tracked_llm_call(
        messages=[{"role": "user", "content": f"Chunk:\n{chunk['content']}"}],
        system=CHUNK_TRACEABLE_GENERATOR_PROMPT
    )
    q_match = re.search(r"QUESTION:\s*(.+)", raw)
    a_match = re.search(r"ANSWER:\s*(.+)", raw)
    if not q_match or not a_match:
        return None
    question, answer = q_match.group(1).strip(), a_match.group(1).strip()
    if question.upper() == "SKIP":
        return None
    # THE FIX: save the exact source chunk text, not just its section label
    return {
        "question": question,
        "ground_truth": answer,
        "source_chunk_content": chunk["content"],
        "source_section": chunk["section"],
    }

benchmark_chunks = sample_chunks_across_sections(chunks_per_section=3)
traceable_pairs = []
for chunk in benchmark_chunks:
    pair = generate_traceable_pair(chunk)
    if pair:
        traceable_pairs.append(pair)
        print(f"✅ [{pair['source_section']}] {pair['question'][:65]}")

print(f"\n📊 Built {len(traceable_pairs)} chunk-traceable benchmark questions")

📣 Top retrieval score : 0.798 (threshold: 0.5)
📣 Top retrieval score : 0.813 (threshold: 0.5)
📣 Top retrieval score : 0.775 (threshold: 0.5)
📣 Top retrieval score : 0.828 (threshold: 0.5)
📣 Top retrieval score : 0.829 (threshold: 0.5)
📣 Top retrieval score : 0.797 (threshold: 0.5)
📋 Sampled 18 chunks across 6 sections
✅ [business overview and products] What types of products does the Company design, manufacture, and 
✅ [business overview and products] To which types of customers does the Company sell its products an
✅ [business overview and products] What are the characteristics of the markets in which the Company’
✅ [risk factors] According to the risk factors section, what does the Company say 
✅ [risk factors] According to the Risk Factors section, which aspects of the Compa
✅ [risk factors] What does the discussion of risk factors contain?
✅ [management discussion and analysis] What makes auditing management’s evaluation of uncertain tax posi
✅ [management discussion and analysis] 

In [7]:
# Building the controlled comparison pool

# The pool every candidate model will be ranked against — includes every
# true source chunk PLUS extra "distractor" chunks, so ranking is meaningful
distractor_chunks = sample_chunks_across_sections(chunks_per_section=4)

chunk_pool = list({c["content"] for c in benchmark_chunks} | {c["content"] for c in distractor_chunks})
print(f"📋 Comparison pool: {len(chunk_pool)} unique chunks "
      f"(includes all {len(traceable_pairs)} true source chunks + distractors)")

📣 Top retrieval score : 0.798 (threshold: 0.5)
📣 Top retrieval score : 0.813 (threshold: 0.5)
📣 Top retrieval score : 0.775 (threshold: 0.5)
📣 Top retrieval score : 0.828 (threshold: 0.5)
📣 Top retrieval score : 0.829 (threshold: 0.5)
📣 Top retrieval score : 0.797 (threshold: 0.5)
📋 Sampled 24 chunks across 6 sections
📋 Comparison pool: 22 unique chunks (includes all 18 true source chunks + distractors)


In [8]:
# Hit Rate@K, Computed Identically for Any Model

def compute_hit_rate(model: SentenceTransformer, model_name: str, k: int = 4) -> dict:
    print(f"\n{'='*60}\nEvaluating: {model_name}\n{'='*60}")

    pool_embeddings = model.encode(chunk_pool, show_progress_bar=True, normalize_embeddings=True)

    hits = 0
    per_question_results = []

    for pair in traceable_pairs:
        query_embedding = model.encode([pair["question"]], normalize_embeddings=True)[0]
        similarities = pool_embeddings @ query_embedding  # cosine similarity, since normalized
        top_k_indices = np.argsort(similarities)[::-1][:k]
        top_k_chunks = [chunk_pool[i] for i in top_k_indices]

        is_hit = pair["source_chunk_content"] in top_k_chunks
        hits += int(is_hit)
        per_question_results.append({"question": pair["question"], "hit": is_hit})
        print(f"  {'✅' if is_hit else '❌'} {pair['question'][:60]}")

    hit_rate = hits / len(traceable_pairs)
    print(f"\n📊 {model_name} — Hit Rate@{k}: {hits}/{len(traceable_pairs)} ({hit_rate*100:.1f}%)")
    return {"model_name": model_name, "hit_rate": hit_rate, "results": per_question_results}

In [9]:
# Running all three candidate models
CANDIDATE_MODELS = {
    "BAAI/bge-m3": "Current production model (multilingual, long-context)",
    "sentence-transformers/all-MiniLM-L6-v2": "Small, fast, general-purpose baseline",
    "BAAI/bge-large-en-v1.5": "Strong English-only alternative",
}

model_results = []
for model_id, description in CANDIDATE_MODELS.items():
    print(f"\n📥 Loading {model_id} — {description}")
    model = SentenceTransformer(model_id)
    result = compute_hit_rate(model, model_id, k=4)
    model_results.append(result)
    del model  # free memory before loading the next one


📥 Loading BAAI/bge-m3 — Current production model (multilingual, long-context)

Evaluating: BAAI/bge-m3


Batches: 100%|██████████| 1/1 [01:22<00:00, 82.04s/it]


  ✅ What types of products does the Company design, manufacture,
  ✅ To which types of customers does the Company sell its produc
  ✅ What are the characteristics of the markets in which the Com
  ✅ According to the risk factors section, what does the Company
  ✅ According to the Risk Factors section, which aspects of the 
  ✅ What does the discussion of risk factors contain?
  ✅ What makes auditing management’s evaluation of uncertain tax
  ✅ What controls did the auditors test in relation to the evalu
  ✅ What factors may this analysis be based on?
  ✅ On which page of the Index to Consolidated Financial Stateme
  ✅ Whose accounts are included in the consolidated financial st
  ✅ On which page of the Index to Consolidated Financial Stateme
  ✅ On which page can the Consolidated Statements of Cash Flows 
  ✅ What was the total other non-current assets reported for 202
  ✅ What were Apple's total non‑current liabilities as of Septem
  ✅ Under what circumstance may payments of principal

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.09s/it]


  ✅ What types of products does the Company design, manufacture,
  ✅ To which types of customers does the Company sell its produc
  ✅ What are the characteristics of the markets in which the Com
  ✅ According to the risk factors section, what does the Company
  ✅ According to the Risk Factors section, which aspects of the 
  ✅ What does the discussion of risk factors contain?
  ✅ What makes auditing management’s evaluation of uncertain tax
  ✅ What controls did the auditors test in relation to the evalu
  ✅ What factors may this analysis be based on?
  ✅ On which page of the Index to Consolidated Financial Stateme
  ✅ Whose accounts are included in the consolidated financial st
  ✅ On which page of the Index to Consolidated Financial Stateme
  ✅ On which page can the Consolidated Statements of Cash Flows 
  ✅ What was the total other non-current assets reported for 202
  ✅ What were Apple's total non‑current liabilities as of Septem
  ✅ Under what circumstance may payments of principal

Batches: 100%|██████████| 1/1 [00:41<00:00, 41.33s/it]


  ✅ What types of products does the Company design, manufacture,
  ✅ To which types of customers does the Company sell its produc
  ✅ What are the characteristics of the markets in which the Com
  ✅ According to the risk factors section, what does the Company
  ✅ According to the Risk Factors section, which aspects of the 
  ✅ What does the discussion of risk factors contain?
  ✅ What makes auditing management’s evaluation of uncertain tax
  ✅ What controls did the auditors test in relation to the evalu
  ✅ What factors may this analysis be based on?
  ✅ On which page of the Index to Consolidated Financial Stateme
  ✅ Whose accounts are included in the consolidated financial st
  ✅ On which page of the Index to Consolidated Financial Stateme
  ✅ On which page can the Consolidated Statements of Cash Flows 
  ✅ What was the total other non-current assets reported for 202
  ✅ What were Apple's total non‑current liabilities as of Septem
  ✅ Under what circumstance may payments of principal

In [10]:
# The comparison Scorecard

print(f"\n{'='*60}")
print("FINAL SCORECARD — Hit Rate@4 across all candidate models")
print(f"{'='*60}")
print(f"{'Model':<40} {'Hit Rate':>10}")
print("-" * 52)
for r in sorted(model_results, key=lambda x: -x["hit_rate"]):
    marker = " ← current production model" if r["model_name"] == "BAAI/bge-m3" else ""
    print(f"{r['model_name']:<40} {r['hit_rate']*100:>9.1f}%{marker}")

current_model_rate = next(r["hit_rate"] for r in model_results if r["model_name"] == "BAAI/bge-m3")
best_model = max(model_results, key=lambda x: x["hit_rate"])

print(f"\n📈 Current model (bge-m3): {current_model_rate*100:.1f}%")
print(f"📈 Best candidate: {best_model['model_name']} at {best_model['hit_rate']*100:.1f}%")

gap = best_model["hit_rate"] - current_model_rate
if gap > 0.1:
    print(f"\n⚠️  Meaningful gap detected ({gap*100:.1f} points) — worth investigating "
          f"a model swap or fine-tuning tomorrow.")
elif gap > 0:
    print(f"\n➖ Small gap ({gap*100:.1f} points) — likely not worth the switching cost alone, "
          f"but keep in mind for Day 7's fine-tuning discussion.")
else:
    print(f"\n✅ Current model already performs as well as or better than the alternatives tested — "
          f"real evidence that fine-tuning may not be the highest-value next step.")

with open("day6_hit_rate_benchmark.json", "w") as f:
    json.dump(model_results, f, indent=2)
print("\n📁 Saved to day6_hit_rate_benchmark.json")


FINAL SCORECARD — Hit Rate@4 across all candidate models
Model                                      Hit Rate
----------------------------------------------------
BAAI/bge-m3                                  100.0% ← current production model
sentence-transformers/all-MiniLM-L6-v2       100.0%
BAAI/bge-large-en-v1.5                       100.0%

📈 Current model (bge-m3): 100.0%
📈 Best candidate: BAAI/bge-m3 at 100.0%

✅ Current model already performs as well as or better than the alternatives tested — real evidence that fine-tuning may not be the highest-value next step.

📁 Saved to day6_hit_rate_benchmark.json
